### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="iranian_churn",
    dataset_year="2011",
    domain_str="business & marketing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5JW3Z",
    download_description="""
mkdir -p local-data-warehouse/iranian_churn \
&& wget -P local-data-warehouse/iranian_churn/ https://archive.ics.uci.edu/static/public/563/iranian+churn+dataset.zip \
&& unzip local-data-warehouse/iranian_churn/iranian+churn+dataset.zip -d local-data-warehouse/iranian_churn/ \
&& rm local-data-warehouse/iranian_churn/iranian+churn+dataset.zip
""",
    # References
    academic_reference_bibtex="""@article{keramati2011churn,
  title={Churn analysis for an Iranian mobile operator},
  author={Keramati, Abbas and Ardabili, Seyed MS},
  journal={Telecommunications Policy},
  volume={35},
  number={4},
  pages={344--356},
  year={2011},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="keramati2011churn",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- The data collection process for the task is not ideal. The data was collected for each individual right up to the churn month. That means, for the churners we observed varying length periods, while for the other individuals we observed the full period.
- Nevertheless, the features are not collected after the churn and the task is still valid, if we conceptualize it as “identify customers close to churn” rather than “predict churn ahead of time”.
- We remove exact duplicates (9.52% of the samples) from the data to avoid leaks. The data contains features like "Seconds of Use", "Frequency of use", "Frequency of SMS", "Distinct Called Numbers", and most importantly a customer value assignment. Given just 3000 samples, it is reasonable to assume that if feature values with so many degrees of freedom are exactly the same, they belong to the same customer.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Churn",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Churn",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "Customer Churn.csv")
df = df.drop_duplicates().reset_index(drop=True)

df["Tariff Plan"] = df["Tariff Plan"].map({1: "Pay as you go", 2: "Contractual"})
df["Status"] = df["Status"].map({1: "Active", 2: "Non-active"})
df["Churn"] = df["Churn"].map({1: "Churn", 0: "Non-churn"})
df["Complains"] = df["Complains"].map({0: "No complaint", 1: "Complaint"})

as_cat_dtype = [
    "Tariff Plan",
    "Complains",
    "Status",
    "Churn",
    # Ordinals which we keep as num
    # "Charge Amount",
    # "Age Group",
]
df[as_cat_dtype] = df[as_cat_dtype].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Loaded data shape:", df.shape)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For IID Data
split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()